### Offshore DAS Dataset

Author: Benz Poobua (spoobua@stanford.edu)

Reference: Williams, E. F., Fernandez-Ruiz, M. R., Magalhaes, R., Vanthillo, R., Zhan, Z., Gonzalez-Herraez, M., & Martins, H. F. (2019). Belgium Distributed Acoustic Sensing Array Raw Data (1.0) [Data set]. CaltechDATA. https://doi.org/10.22002/D1.1296

### Offshore DAS Processing

Reference: Williams, E. F., Fernández-Ruiz, M. R., Magalhaes, R., Vanthillo, R., Zhan, Z., González-Herráez, M., & Martins, H. F. (2021). Scholte wave inversion and passive source imaging with ocean-bottom DAS. The Leading Edge, 40(8), 576-583.doi: https://doi.org/10.1190/tle40080576.1

**Source Change**
* Part I: Focused on low-frequency Ocean Gravity Waves ($0.1–0.3$ Hz).

* Part II: Focuses on high-frequency Scholte Waves ($1.0–5.0$ Hz). 

**Key foundings**
* They found "anomalous" signals that didn't match the standard background noise. By "migrating" (back-tracing) these waves, they discovered they were coming from individual wind turbines.

* Therefore, noise isn't actually random; it's coming from fixed points (the turbines). This explains why their cross-correlations (CC) converge so fast—only one hour of data—because the "source" is a massive industrial machine constantly vibrating.

* Scholte Wave Inversion: Between 0.3 and 5 Hz, the background noise consists of Scholte waves.

**Why Shear-Wave Velocity ($V_S$) Matters**
* Hazard Assessment: $V_S$ is used to estimate seismic and liquefaction hazards—the risk of the soil turning into liquid during an earthquake (Abrahamson and Silva, 2008).

* Geotechnical Design: It identifies "small-strain stiffness," which is critical for the foundations of massive structures like wind turbines (Seed and Idriss, 1970).

* Production Monitoring: Changes in $V_S$ over time can show how the ground is compacting or deforming due to oil/gas extraction (Hatchell et al., 2009).

**Terminology**
* $V_{S30}$: The average shear-wave velocity in the top 30 meters of soil -- for classifying how a site will respond to earthquake shaking.

* $Z_{1.0}$: The depth at which the shear-wave velocity reaches 1.0 km/s -- defining the thickness of the "soft" sediment layer before hitting harder rock.

**Site Context**
* Bathymetry: The water is shallow (< 40 m) and characterized by large sand ridges. 

* Cable Burial: The fiber is buried 0.5 to 3.5 m deep. It provides better "coupling" to the ground, allowing it to pick up seismic Scholte waves more clearly.

**Instrumental Specs: Chirped-Pulse DAS**

Conventional DAS often suffers from "fading," where certain parts of the fiber randomly become insensitive to vibration. This system eliminates that, meaning we can trust the raw amplitude of the signal at every single channel without needing complex recalibration (Fernandez-Ruiz et al., 2018).

|Parameter|Value|Importance|
|--|--|--|
|fs|10.0 Hz|Determines frequency axis and Nyquist limit (5 Hz).
|dx|10.0 m|Used to calculate the distance and wavenumber ($k$) axes.
|Gauge Length|10 m|Acts as a spatial filter; signals shorter than 10m are "blurred."|
|Target Band|0.3 – 5.0 Hz|This is where the Scholte waves live for this site.|
|Source Type|Ambient / Turbine|Use Cross-Correlation to turn these into virtual shots.|

**(Their) Workflow**
* Windowing: The data was sliced into 3.4-minute segments ($2048$ samples). Overlapping these by 50% helps smooth the results.

* Preprocessing: They applied Spectral Whitening. This is a normalization step that ensures all frequencies (from $0.3$ to $5$ Hz) have equal "weight," preventing the loud ocean hum from drowning out the higher-frequency Scholte waves.

* Cross-Correlation (CC): They correlated the signals between channels to create Virtual Source Gathers (VSG).

* The $\tau-p$ Transform: To turn the VSG into a "Dispersion Image" (a map of velocity vs. frequency), they used the $\tau-p$ transform.

**The Power-Law Inversion**

Instead of modeling the seafloor as distinct, flat layers of rock and mud, they used a continuous power-law parameterization:$$c(z) = c_0 z^\nu$$Where $c_0$ is the surface velocity, $z$ is depth, and $\nu$ is the gradient.

Marine sediments get packed tighter by the pressure of the water and soil above them. Theoretical and experimental studies show that this "confining pressure" naturally causes shear-wave velocity to increase following a power-law curve in the top tens of meters (Hamilton, 1976; Bryan and Stoll, 1988; Buckingham, 2005).

* $V_{S30}$: The average shear-wave velocity of the top 30 meters. Formula: $c_0(1 - \nu)30^\nu$

* $Z_{1.0}$: The depth where the sediment gets hard enough that waves travel at 1 km/s. Formula: $(1000/c_0)^{1/\nu}$

In [ ]:
import os 
import numpy as np
import scipy.io as sio
from tqdm import tqdm
import datetime
import re

In [ ]:
# Define paths
data_dir = os.path.join('..', 'data', 'raw_offshore')
input_filename = 'mat_2018_08_19_00h28m05s_Parkwind_HDAS_2Dmap_StrainData_2D.mat'
mat_path = os.path.join(data_dir, input_filename)

# Parse Start Time from Filename using Regex
# Looks for: YYYY_MM_DD_HHhMMmSSs
match = re.search(r'(\d{4})_(\d{2})_(\d{2})_(\d{2})h(\d{2})m(\d{2})s', input_filename)
if match:
    year, month, day, hour, minute, second = map(int, match.groups())
    start_time = datetime.datetime(year, month, day, hour, minute, second)
    print(f"Detected start time: {start_time}")
else:
    # Fallback if regex fails
    start_time = datetime.datetime(2018, 8, 19, 0, 28, 5)
    print("Warning: Could not parse time from filename. Using fallback.")

# Slicing parameters (from Williams et al., 2021)
n_window = 2048      # 3.4 minutes at 10 Hz
n_step = 1024        # 50% overlap

# Metadata 
dx = 10.0            # Channel spacing in meters
fs = 10.0            # Sampling frequency in Hz
dt = 1.0 / fs        # The critical key required by utils.py

In [ ]:
# Load the .mat file
print(f"Loading large .mat file: {input_filename}...")
mat_contents = sio.loadmat(mat_path)
strain_data = mat_contents['Data_2D']

# Discard the first 1000 channels (surf zone)
usable_strain = strain_data[1000:, :]
num_channels, total_samples = usable_strain.shape
print(f"Data ready. Shape after dropping surf zone (nch x nt): {usable_strain.shape}")

In [ ]:
# Slicing loop 
window_idx = 0
start_sample = 0
total_expected_windows = (total_samples - n_window) // n_step + 1

with tqdm(total=total_expected_windows, desc="Slicing Windows", unit="win") as pbar:
    while (start_sample + n_window) <= total_samples:
        end_sample = start_sample + n_window
        
        # A. Calculate precise timestamp for this window
        # Offset = start_sample / sampling_rate
        time_offset = datetime.timedelta(seconds=(start_sample / fs))
        current_window_time = start_time + time_offset
        
        # B. Create the explanatory, stack-friendly filename
        # Format: 20180819_002805_win0000_offshore.npz
        time_str = current_window_time.strftime("%Y%m%d_%H%M%S")
        out_filename = os.path.join(data_dir, f'{time_str}_win{window_idx:04d}_offshore.npz')
        
        # C. Process and Save
        raw_slice = usable_strain[:, start_sample:end_sample].copy()
        raw_slice -= np.mean(raw_slice, axis=-1, keepdims=True)
        
        np.savez_compressed(
            out_filename,
            data=raw_slice.astype(np.float32),
            dt=dt,                             
            start_sample=start_sample,            
            end_sample=end_sample
        )
        
        # Advance
        start_sample += n_step
        window_idx += 1
        pbar.update(1)

print(f"\nSuccess! Created {window_idx} windows.")
print(f"Last file saved: {out_filename}")